In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import warnings
from imblearn.over_sampling import SMOTE
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
import pickle
warnings.filterwarnings('ignore')

# config visualisasi
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# # Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Config path dataset bersih
BASE_PATH = "/content/drive/MyDrive/DeepWell/"
PROC_PATH = os.path.join(BASE_PATH, "data/processed/")
TRANS_PATH = os.path.join(BASE_PATH, "data/transformed/")

print("[SUCCESS] Initialization complete. Libraries imported and visual settings configured.")

Mounted at /content/drive
[SUCCESS] Initialization complete. Libraries imported and visual settings configured.


### Inisialisasi dan Penyiapan

Sel ini melakukan penyiapan awal untuk notebook. Ini mengimpor pustaka yang diperlukan seperti `pandas` untuk manipulasi data, `numpy` untuk operasi numerik, `matplotlib.pyplot` dan `seaborn` untuk visualisasi data, dan `scipy.stats` untuk fungsi statistik. Ini juga mengkonfigurasi gaya visual `seaborn` dan ukuran serta font gambar `matplotlib`. Peringatan difilter untuk menjaga output tetap bersih.

Pentingnya, ia memasang Google Drive untuk mengakses file dan menyiapkan jalur dasar untuk pemuatan data.

**Hasil:**
- Google Drive berhasil dipasang, ditunjukkan oleh `Drive already mounted at /content/drive`.
- Pesan `[SUCCESS] Initialization complete. Libraries imported and visual settings configured.` menegaskan bahwa semua penyiapan awal telah berhasil dieksekusi tanpa masalah.

In [2]:
# Load dataset

files = {
    "sleep": "sleep_health_clean.csv",
    "mental": "mental_health_clean.csv",
    "stress": "student_stress_clean.csv"
}

dfs = {}

for key, filename in files.items():
  path = os.path.join(PROC_PATH, filename)
  if os.path.exists(path):
    dfs[key] = pd.read_csv(path)
    print(f"[SUCCESS] Dataset '{key}' successfully loaded from: {path}")
  else:
    print(f"[ERROR] File not found: {path}")

df_sleep = dfs.get("sleep")
df_mental = dfs.get("mental")
df_stress = dfs.get("stress")

print("\n[SUCCESS] All datasets are ready for EDA.")

[SUCCESS] Dataset 'sleep' successfully loaded from: /content/drive/MyDrive/DeepWell/data/processed/sleep_health_clean.csv
[SUCCESS] Dataset 'mental' successfully loaded from: /content/drive/MyDrive/DeepWell/data/processed/mental_health_clean.csv
[SUCCESS] Dataset 'stress' successfully loaded from: /content/drive/MyDrive/DeepWell/data/processed/student_stress_clean.csv

[SUCCESS] All datasets are ready for EDA.


In [3]:
df_sleep

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,BP_Systolic,BP_Diastolic
0,Male,41,Nurse,7.0,5,91,2,Normal,56,9476,NaN,111.0,71.15
1,Male,42,Software Engineer,6.7,7,123,1,Overweight,67,10661,NaN,129.0,85.85
2,Female,45,Doctor,8.7,4,49,1,Normal,60,5033,NaN,119.0,79.35
3,Male,24,Writer,8.0,6,4,1,Normal,97,1610,NaN,98.0,62.70
4,Female,18,Salesperson,8.8,6,66,1,Overweight,71,6574,NaN,110.0,66.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,Male,63,Engineer,9.1,7,34,1,Obese,70,4830,NaN,131.0,90.15
1496,Male,70,Lawyer,8.6,1,26,2,Normal,104,3576,Narcolepsy,117.0,74.05
1497,Male,23,Engineer,7.3,5,62,2,Overweight,67,6785,Sleep Apnea,110.0,67.50
1498,Male,76,Engineer,8.1,7,36,1,Overweight,54,3751,Insomnia,126.0,86.90


In [4]:
df_mental

,Age,Gender,Exercise Level,Diet Type,Sleep Hours,Stress Level,Mental Health Condition,Work Hours per Week,Screen Time per Day (Hours),Social Interaction Score,Happiness Score
0,48,Male,Low,Vegetarian,6.3,0,NaN,21,4.0,7.8,6.5
1,31,Male,Moderate,Vegan,4.9,0,Ptsd,48,5.2,8.2,6.8
2,37,Female,Low,Vegetarian,7.2,2,NaN,43,4.7,9.6,9.7
3,35,Male,Low,Vegan,7.2,0,Depression,43,2.2,8.2,6.6
4,46,Male,Low,Balanced,7.3,0,Anxiety,35,3.6,4.7,4.4
...,...,...,...,...,...,...,...,...,...,...,...
2995,57,Female,Moderate,Balanced,7.0,2,Depression,29,4.4,9.7,5.9
2996,27,Male,Low,Junk Food,7.1,0,NaN,47,7.4,6.3,9.9
2997,42,Male,Moderate,Balanced,6.0,2,Depression,23,3.9,5.2,4.1
2998,25,Male,High,Keto,5.7,0,Anxiety,51,4.3,5.9,4.1


In [5]:
df_stress

,anxiety_level,self_esteem,mental_health_history,depression,headache,blood_pressure,sleep_quality,breathing_problem,noise_level,living_conditions,safety,basic_needs,social_support,Stress Level
0,14,20,0,11,2,1,2,4,2,3,3,2,2,1
1,15,8,1,15,5,3,1,4,3,1,2,2,1,2
2,12,18,1,14,2,1,2,2,2,2,3,2,2,1
3,16,12,1,15,4,3,1,3,4,2,2,2,1,2
4,16,28,0,7,2,3,5,1,3,2,4,3,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1093,11,17,0,14,3,1,3,2,2,2,2,3,3,1
1094,9,12,0,8,0,3,0,0,0,1,3,4,1,2
1095,4,26,0,3,1,2,5,2,2,3,4,4,3,0
1096,21,0,1,19,5,3,1,4,3,1,1,1,1,2


### Data Berdasarkan Frontend

In [6]:
profil = ['gender', 'age', 'weight', 'height', 'long_term_goal']
daily = ['mood', 'sleep_quality', 'sleep_duration', 'activity_level', 'screen_time', 'stress_level']

## Feature Selection

### Sleep

In [7]:
df_sleep

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,BP_Systolic,BP_Diastolic
0,Male,41,Nurse,7.0,5,91,2,Normal,56,9476,NaN,111.0,71.15
1,Male,42,Software Engineer,6.7,7,123,1,Overweight,67,10661,NaN,129.0,85.85
2,Female,45,Doctor,8.7,4,49,1,Normal,60,5033,NaN,119.0,79.35
3,Male,24,Writer,8.0,6,4,1,Normal,97,1610,NaN,98.0,62.70
4,Female,18,Salesperson,8.8,6,66,1,Overweight,71,6574,NaN,110.0,66.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,Male,63,Engineer,9.1,7,34,1,Obese,70,4830,NaN,131.0,90.15
1496,Male,70,Lawyer,8.6,1,26,2,Normal,104,3576,Narcolepsy,117.0,74.05
1497,Male,23,Engineer,7.3,5,62,2,Overweight,67,6785,Sleep Apnea,110.0,67.50
1498,Male,76,Engineer,8.1,7,36,1,Overweight,54,3751,Insomnia,126.0,86.90


In [8]:
df_sleep = df_sleep.drop(columns=['Physical Activity Level', 'Heart Rate', 'Daily Steps', 'BP_Systolic', 'BP_Diastolic'])
df_sleep

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Stress Level,BMI Category,Sleep Disorder
0,Male,41,Nurse,7.0,5,2,Normal,NaN
1,Male,42,Software Engineer,6.7,7,1,Overweight,NaN
2,Female,45,Doctor,8.7,4,1,Normal,NaN
3,Male,24,Writer,8.0,6,1,Normal,NaN
4,Female,18,Salesperson,8.8,6,1,Overweight,NaN
...,...,...,...,...,...,...,...,...
1495,Male,63,Engineer,9.1,7,1,Obese,NaN
1496,Male,70,Lawyer,8.6,1,2,Normal,Narcolepsy
1497,Male,23,Engineer,7.3,5,2,Overweight,Sleep Apnea
1498,Male,76,Engineer,8.1,7,1,Overweight,Insomnia


In [9]:
df_sleep.columns = ['gender', 'age', 'occupation', 'sleep_duration', 'sleep_quality', 'stress_level', 'bmi_cat', 'sleep_disorder']
df_sleep

,gender,age,occupation,sleep_duration,sleep_quality,stress_level,bmi_cat,sleep_disorder
0,Male,41,Nurse,7.0,5,2,Normal,NaN
1,Male,42,Software Engineer,6.7,7,1,Overweight,NaN
2,Female,45,Doctor,8.7,4,1,Normal,NaN
3,Male,24,Writer,8.0,6,1,Normal,NaN
4,Female,18,Salesperson,8.8,6,1,Overweight,NaN
...,...,...,...,...,...,...,...,...
1495,Male,63,Engineer,9.1,7,1,Obese,NaN
1496,Male,70,Lawyer,8.6,1,2,Normal,Narcolepsy
1497,Male,23,Engineer,7.3,5,2,Overweight,Sleep Apnea
1498,Male,76,Engineer,8.1,7,1,Overweight,Insomnia


In [10]:
df_sleep['sleep_disorder'] = df_sleep['sleep_disorder'].fillna('No')
df_sleep

,gender,age,occupation,sleep_duration,sleep_quality,stress_level,bmi_cat,sleep_disorder
0,Male,41,Nurse,7.0,5,2,Normal,No
1,Male,42,Software Engineer,6.7,7,1,Overweight,No
2,Female,45,Doctor,8.7,4,1,Normal,No
3,Male,24,Writer,8.0,6,1,Normal,No
4,Female,18,Salesperson,8.8,6,1,Overweight,No
...,...,...,...,...,...,...,...,...
1495,Male,63,Engineer,9.1,7,1,Obese,No
1496,Male,70,Lawyer,8.6,1,2,Normal,Narcolepsy
1497,Male,23,Engineer,7.3,5,2,Overweight,Sleep Apnea
1498,Male,76,Engineer,8.1,7,1,Overweight,Insomnia


bmi_cat yang tidak diinput oleh user maka akan dilakukan feature engineering dibagian backend

Menghasilkan Feature Selection df_sleep

### Mental Health

In [11]:
df_mental

,Age,Gender,Exercise Level,Diet Type,Sleep Hours,Stress Level,Mental Health Condition,Work Hours per Week,Screen Time per Day (Hours),Social Interaction Score,Happiness Score
0,48,Male,Low,Vegetarian,6.3,0,NaN,21,4.0,7.8,6.5
1,31,Male,Moderate,Vegan,4.9,0,Ptsd,48,5.2,8.2,6.8
2,37,Female,Low,Vegetarian,7.2,2,NaN,43,4.7,9.6,9.7
3,35,Male,Low,Vegan,7.2,0,Depression,43,2.2,8.2,6.6
4,46,Male,Low,Balanced,7.3,0,Anxiety,35,3.6,4.7,4.4
...,...,...,...,...,...,...,...,...,...,...,...
2995,57,Female,Moderate,Balanced,7.0,2,Depression,29,4.4,9.7,5.9
2996,27,Male,Low,Junk Food,7.1,0,NaN,47,7.4,6.3,9.9
2997,42,Male,Moderate,Balanced,6.0,2,Depression,23,3.9,5.2,4.1
2998,25,Male,High,Keto,5.7,0,Anxiety,51,4.3,5.9,4.1


In [12]:
df_mental = df_mental.drop(columns=['Diet Type', 'Stress Level', 'Social Interaction Score'])
df_mental

,Age,Gender,Exercise Level,Sleep Hours,Mental Health Condition,Work Hours per Week,Screen Time per Day (Hours),Happiness Score
0,48,Male,Low,6.3,NaN,21,4.0,6.5
1,31,Male,Moderate,4.9,Ptsd,48,5.2,6.8
2,37,Female,Low,7.2,NaN,43,4.7,9.7
3,35,Male,Low,7.2,Depression,43,2.2,6.6
4,46,Male,Low,7.3,Anxiety,35,3.6,4.4
...,...,...,...,...,...,...,...,...
2995,57,Female,Moderate,7.0,Depression,29,4.4,5.9
2996,27,Male,Low,7.1,NaN,47,7.4,9.9
2997,42,Male,Moderate,6.0,Depression,23,3.9,4.1
2998,25,Male,High,5.7,Anxiety,51,4.3,4.1


In [13]:
df_mental.columns = ['age', 'gender', 'exercise_level', 'sleep_duration', 'mental_health_condition', 'work_hours_per_week', 'screen_time', 'happiness_score']
df_mental

,age,gender,exercise_level,sleep_duration,mental_health_condition,work_hours_per_week,screen_time,happiness_score
0,48,Male,Low,6.3,NaN,21,4.0,6.5
1,31,Male,Moderate,4.9,Ptsd,48,5.2,6.8
2,37,Female,Low,7.2,NaN,43,4.7,9.7
3,35,Male,Low,7.2,Depression,43,2.2,6.6
4,46,Male,Low,7.3,Anxiety,35,3.6,4.4
...,...,...,...,...,...,...,...,...
2995,57,Female,Moderate,7.0,Depression,29,4.4,5.9
2996,27,Male,Low,7.1,NaN,47,7.4,9.9
2997,42,Male,Moderate,6.0,Depression,23,3.9,4.1
2998,25,Male,High,5.7,Anxiety,51,4.3,4.1


In [14]:
df_mental['mental_health_condition'] = df_mental['mental_health_condition'].fillna('No')
df_mental

,age,gender,exercise_level,sleep_duration,mental_health_condition,work_hours_per_week,screen_time,happiness_score
0,48,Male,Low,6.3,No,21,4.0,6.5
1,31,Male,Moderate,4.9,Ptsd,48,5.2,6.8
2,37,Female,Low,7.2,No,43,4.7,9.7
3,35,Male,Low,7.2,Depression,43,2.2,6.6
4,46,Male,Low,7.3,Anxiety,35,3.6,4.4
...,...,...,...,...,...,...,...,...
2995,57,Female,Moderate,7.0,Depression,29,4.4,5.9
2996,27,Male,Low,7.1,No,47,7.4,9.9
2997,42,Male,Moderate,6.0,Depression,23,3.9,4.1
2998,25,Male,High,5.7,Anxiety,51,4.3,4.1


Menghasilkan Feature Selection df_mental

### Stress

In [15]:
df_stress

,anxiety_level,self_esteem,mental_health_history,depression,headache,blood_pressure,sleep_quality,breathing_problem,noise_level,living_conditions,safety,basic_needs,social_support,Stress Level
0,14,20,0,11,2,1,2,4,2,3,3,2,2,1
1,15,8,1,15,5,3,1,4,3,1,2,2,1,2
2,12,18,1,14,2,1,2,2,2,2,3,2,2,1
3,16,12,1,15,4,3,1,3,4,2,2,2,1,2
4,16,28,0,7,2,3,5,1,3,2,4,3,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1093,11,17,0,14,3,1,3,2,2,2,2,3,3,1
1094,9,12,0,8,0,3,0,0,0,1,3,4,1,2
1095,4,26,0,3,1,2,5,2,2,3,4,4,3,0
1096,21,0,1,19,5,3,1,4,3,1,1,1,1,2


## Feature Engneering

### Hanya dilakukan di Backend bukan di train


Sleep

In [16]:
# # Menghitung BMI untuk df sleep
# df_sleep['bmi'] = df_sleep['weight'] / ((df_sleep['height'] / 100) ** 2)

# # Fungsi kategori BMI
# def kategori_bmi(bmi):
#     if bmi < 18.5:
#         return "Underweight"
#     elif bmi < 25:
#         return "Normal"
#     elif bmi < 30:
#         return "Overweight"
#     else:
#         return "Obese"

# # Membuat kolom kategori BMI
# df_sleep['bmi_cat'] = df_sleep['bmi'].apply(kategori_bmi)

# # Menampilkan hasil
# print(df_sleep[['weight', 'height', 'bmi', 'bmi_cat']])

### Dilakukan di Train dan Backend

Sleep

In [17]:
df_sleep['sleep_efficiency'] = df_sleep['sleep_duration'] / 8

In [18]:
df_sleep

,gender,age,occupation,sleep_duration,sleep_quality,stress_level,bmi_cat,sleep_disorder,sleep_efficiency
0,Male,41,Nurse,7.0,5,2,Normal,No,0.8750
1,Male,42,Software Engineer,6.7,7,1,Overweight,No,0.8375
2,Female,45,Doctor,8.7,4,1,Normal,No,1.0875
3,Male,24,Writer,8.0,6,1,Normal,No,1.0000
4,Female,18,Salesperson,8.8,6,1,Overweight,No,1.1000
...,...,...,...,...,...,...,...,...,...
1495,Male,63,Engineer,9.1,7,1,Obese,No,1.1375
1496,Male,70,Lawyer,8.6,1,2,Normal,Narcolepsy,1.0750
1497,Male,23,Engineer,7.3,5,2,Overweight,Sleep Apnea,0.9125
1498,Male,76,Engineer,8.1,7,1,Overweight,Insomnia,1.0125


Mental

In [19]:
df_mental['sleep_efficiency'] = df_mental['sleep_duration'] / 8

In [20]:
df_mental

,age,gender,exercise_level,sleep_duration,mental_health_condition,work_hours_per_week,screen_time,happiness_score,sleep_efficiency
0,48,Male,Low,6.3,No,21,4.0,6.5,0.7875
1,31,Male,Moderate,4.9,Ptsd,48,5.2,6.8,0.6125
2,37,Female,Low,7.2,No,43,4.7,9.7,0.9000
3,35,Male,Low,7.2,Depression,43,2.2,6.6,0.9000
4,46,Male,Low,7.3,Anxiety,35,3.6,4.4,0.9125
...,...,...,...,...,...,...,...,...,...
2995,57,Female,Moderate,7.0,Depression,29,4.4,5.9,0.8750
2996,27,Male,Low,7.1,No,47,7.4,9.9,0.8875
2997,42,Male,Moderate,6.0,Depression,23,3.9,4.1,0.7500
2998,25,Male,High,5.7,Anxiety,51,4.3,4.1,0.7125


## Melakukan Encoding

In [21]:
# Identify categorical columns
categorical_cols_sleep = ['gender', 'occupation', 'bmi_cat']

# Initialize OneHotEncoder
# handle_unknown='ignore' will set unknown categories to all zeros, preventing errors during prediction
encoder_sleep = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
encoded_features_sleep = encoder_sleep.fit_transform(df_sleep[categorical_cols_sleep])

# Create a DataFrame from the encoded features with appropriate column names
encoded_df_sleep = pd.DataFrame(encoded_features_sleep, columns=encoder_sleep.get_feature_names_out(categorical_cols_sleep))

# Drop original categorical columns from df_sleep and concatenate with encoded features
df_sleep_engineered = pd.concat([
    df_sleep.drop(columns=categorical_cols_sleep),
    encoded_df_sleep
], axis=1)

print("Shape of original df_sleep:", df_sleep.shape)
print("Shape of engineered df_sleep:", df_sleep_engineered.shape)
print("\nFirst 5 rows of engineered df_sleep:")
display(df_sleep_engineered.head())


Shape of original df_sleep: (1500, 9)
Shape of engineered df_sleep: (1500, 27)

First 5 rows of engineered df_sleep:


,age,sleep_duration,sleep_quality,stress_level,sleep_disorder,sleep_efficiency,gender_Female,gender_Male,occupation_Accountant,occupation_Artist,...,occupation_Salesperson,occupation_Scientist,occupation_Software Engineer,occupation_Student,occupation_Teacher,occupation_Writer,bmi_cat_Normal,bmi_cat_Obese,bmi_cat_Overweight,bmi_cat_Underweight
0,41,7.0,5,2,No,0.8750,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,42,6.7,7,1,No,0.8375,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,45,8.7,4,1,No,1.0875,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,24,8.0,6,1,No,1.0000,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
4,18,8.8,6,1,No,1.1000,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Melakukan encoding pada data df_sleep

Mental

In [22]:
df_mental

,age,gender,exercise_level,sleep_duration,mental_health_condition,work_hours_per_week,screen_time,happiness_score,sleep_efficiency
0,48,Male,Low,6.3,No,21,4.0,6.5,0.7875
1,31,Male,Moderate,4.9,Ptsd,48,5.2,6.8,0.6125
2,37,Female,Low,7.2,No,43,4.7,9.7,0.9000
3,35,Male,Low,7.2,Depression,43,2.2,6.6,0.9000
4,46,Male,Low,7.3,Anxiety,35,3.6,4.4,0.9125
...,...,...,...,...,...,...,...,...,...
2995,57,Female,Moderate,7.0,Depression,29,4.4,5.9,0.8750
2996,27,Male,Low,7.1,No,47,7.4,9.9,0.8875
2997,42,Male,Moderate,6.0,Depression,23,3.9,4.1,0.7500
2998,25,Male,High,5.7,Anxiety,51,4.3,4.1,0.7125


In [23]:
from sklearn.preprocessing import OneHotEncoder

# Identify categorical columns in df_mental
categorical_cols_mental = ['gender', 'exercise_level', 'mental_health_condition']

# Initialize OneHotEncoder
# handle_unknown='ignore' will set unknown categories to all zeros, preventing errors during prediction
encoder_mental = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
encoded_features_mental = encoder_mental.fit_transform(df_mental[categorical_cols_mental])

# Create a DataFrame from the encoded features with appropriate column names
encoded_df_mental = pd.DataFrame(encoded_features_mental, columns=encoder_mental.get_feature_names_out(categorical_cols_mental))

# Drop original categorical columns from df_mental and concatenate with encoded features
df_mental_engineered = pd.concat([
    df_mental.drop(columns=categorical_cols_mental),
    encoded_df_mental
], axis=1)

print("Shape of original df_mental:", df_mental.shape)
print("Shape of engineered df_mental:", df_mental_engineered.shape)
print("\nFirst 5 rows of engineered df_mental:")
display(df_mental_engineered.head())

Shape of original df_mental: (3000, 9)
Shape of engineered df_mental: (3000, 17)

First 5 rows of engineered df_mental:


,age,sleep_duration,work_hours_per_week,screen_time,happiness_score,sleep_efficiency,gender_Female,gender_Male,gender_Other,exercise_level_High,exercise_level_Low,exercise_level_Moderate,mental_health_condition_Anxiety,mental_health_condition_Bipolar,mental_health_condition_Depression,mental_health_condition_No,mental_health_condition_Ptsd
0,48,6.3,21,4.0,6.5,0.7875,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,31,4.9,48,5.2,6.8,0.6125,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,37,7.2,43,4.7,9.7,0.9000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,35,7.2,43,2.2,6.6,0.9000,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,46,7.3,35,3.6,4.4,0.9125,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


Melakukan encoding pada data df_mental

## Train test and Scalling

### Sleep

In [24]:
# Define features (X) and target (y)
X_sleep = df_sleep_engineered.drop('sleep_disorder', axis=1)
y_sleep = df_sleep_engineered['sleep_disorder']

# Train-test split
X_train_sleep, X_test_sleep, y_train_sleep, y_test_sleep = train_test_split(
    X_sleep,
    y_sleep,
    test_size=0.2,
    random_state=42,
    stratify=y_sleep
)

# Standard Scaling
scaler_sleep = StandardScaler()

X_train_sleep_scaled = scaler_sleep.fit_transform(X_train_sleep)
X_test_sleep_scaled = scaler_sleep.transform(X_test_sleep)

# Convert back to DataFrame
X_train_sleep_scaled = pd.DataFrame(
    X_train_sleep_scaled,
    columns=X_sleep.columns,
    index=X_train_sleep.index
)

X_test_sleep_scaled = pd.DataFrame(
    X_test_sleep_scaled,
    columns=X_sleep.columns,
    index=X_test_sleep.index
)
print("Shape of X_train_sleep_scaled:", X_train_sleep_scaled.shape)
print("Shape of X_test_sleep_scaled:", X_test_sleep_scaled.shape)
print("Shape of y_train_sleep:", y_train_sleep.shape)
print("Shape of y_test_sleep:", y_test_sleep.shape)

print("\nFirst 5 rows of scaled X_train_sleep:")
display(X_train_sleep_scaled.head())

smote = SMOTE(random_state=42)

X_train_sleep_smote, y_train_sleep_smote = smote.fit_resample(
    X_train_sleep_scaled,
    y_train_sleep
)

# Convert to DataFrame
X_train_sleep_smote = pd.DataFrame(
    X_train_sleep_smote,
    columns=X_sleep.columns
)

# Checking shape
print("Before SMOTE:")
print(y_train_sleep.value_counts())
print("\nAfter SMOTE:")
print(pd.Series(y_train_sleep_smote).value_counts())
print("\nShape of X_train_sleep_smote:", X_train_sleep_smote.shape)
print("Shape of y_train_sleep_smote:", y_train_sleep_smote.shape)
print("\nFirst 5 rows of SMOTE data:")
display(X_train_sleep_smote.head())

Shape of X_train_sleep_scaled: (1200, 26)
Shape of X_test_sleep_scaled: (300, 26)
Shape of y_train_sleep: (1200,)
Shape of y_test_sleep: (300,)

First 5 rows of scaled X_train_sleep:


,age,sleep_duration,sleep_quality,stress_level,sleep_efficiency,gender_Female,gender_Male,occupation_Accountant,occupation_Artist,occupation_Chef,...,occupation_Salesperson,occupation_Scientist,occupation_Software Engineer,occupation_Student,occupation_Teacher,occupation_Writer,bmi_cat_Normal,bmi_cat_Obese,bmi_cat_Overweight,bmi_cat_Underweight
1479,-1.027935,0.291866,0.107334,-0.218573,0.291866,-1.033908,1.033908,-0.279584,3.792706,-0.261852,...,-0.276104,-0.270823,-0.254507,-0.254507,-0.269047,-0.258199,-0.785653,-0.511693,1.466154,-0.320844
935,-1.355927,-0.152318,0.669781,1.297538,-0.152318,-1.033908,1.033908,-0.279584,-0.263664,-0.261852,...,-0.276104,-0.270823,-0.254507,-0.254507,-0.269047,-0.258199,1.272827,-0.511693,-0.682057,-0.320844
316,-1.410592,-0.374410,-1.017561,-0.218573,-0.374410,-1.033908,1.033908,-0.279584,-0.263664,-0.261852,...,-0.276104,-0.270823,3.929167,-0.254507,-0.269047,-0.258199,-0.785653,-0.511693,1.466154,-0.320844
233,-1.082600,1.069187,0.107334,-0.218573,1.069187,0.967204,-0.967204,-0.279584,-0.263664,3.818955,...,-0.276104,-0.270823,-0.254507,-0.254507,-0.269047,-0.258199,1.272827,-0.511693,-0.682057,-0.320844
1428,0.393362,-0.818593,-0.455114,-0.218573,-0.818593,0.967204,-0.967204,-0.279584,-0.263664,-0.261852,...,-0.276104,-0.270823,-0.254507,-0.254507,-0.269047,-0.258199,-0.785653,-0.511693,1.466154,-0.320844


Before SMOTE:
sleep_disorder
No                       769
Sleep Apnea              142
Insomnia                 137
Restless Leg Syndrome     82
Narcolepsy                70
Name: count, dtype: int64

After SMOTE:
sleep_disorder
No                       769
Insomnia                 769
Restless Leg Syndrome    769
Narcolepsy               769
Sleep Apnea              769
Name: count, dtype: int64

Shape of X_train_sleep_smote: (3845, 26)
Shape of y_train_sleep_smote: (3845,)

First 5 rows of SMOTE data:


,age,sleep_duration,sleep_quality,stress_level,sleep_efficiency,gender_Female,gender_Male,occupation_Accountant,occupation_Artist,occupation_Chef,...,occupation_Salesperson,occupation_Scientist,occupation_Software Engineer,occupation_Student,occupation_Teacher,occupation_Writer,bmi_cat_Normal,bmi_cat_Obese,bmi_cat_Overweight,bmi_cat_Underweight
0,-1.027935,0.291866,0.107334,-0.218573,0.291866,-1.033908,1.033908,-0.279584,3.792706,-0.261852,...,-0.276104,-0.270823,-0.254507,-0.254507,-0.269047,-0.258199,-0.785653,-0.511693,1.466154,-0.320844
1,-1.355927,-0.152318,0.669781,1.297538,-0.152318,-1.033908,1.033908,-0.279584,-0.263664,-0.261852,...,-0.276104,-0.270823,-0.254507,-0.254507,-0.269047,-0.258199,1.272827,-0.511693,-0.682057,-0.320844
2,-1.410592,-0.374410,-1.017561,-0.218573,-0.374410,-1.033908,1.033908,-0.279584,-0.263664,-0.261852,...,-0.276104,-0.270823,3.929167,-0.254507,-0.269047,-0.258199,-0.785653,-0.511693,1.466154,-0.320844
3,-1.082600,1.069187,0.107334,-0.218573,1.069187,0.967204,-0.967204,-0.279584,-0.263664,3.818955,...,-0.276104,-0.270823,-0.254507,-0.254507,-0.269047,-0.258199,1.272827,-0.511693,-0.682057,-0.320844
4,0.393362,-0.818593,-0.455114,-0.218573,-0.818593,0.967204,-0.967204,-0.279584,-0.263664,-0.261852,...,-0.276104,-0.270823,-0.254507,-0.254507,-0.269047,-0.258199,-0.785653,-0.511693,1.466154,-0.320844


Melakukann train test split 80:20 dan melakukan scalling pada variabel x

### Mental

In [25]:
# Define features (X) and target (y) for mental health data
X_mental = df_mental_engineered.drop(['happiness_score'], axis=1)
y_mental = df_mental_engineered[['happiness_score']]

# Perform train-test split
X_train_mental, X_test_mental, y_train_mental, y_test_mental = train_test_split(X_mental, y_mental, test_size=0.2, random_state=42)

# Initialize StandardScaler
scaler_mental = StandardScaler()

# Fit the scaler on the training data and transform both training and test data
X_train_mental_scaled = scaler_mental.fit_transform(X_train_mental)
X_test_mental_scaled = scaler_mental.transform(X_test_mental)

# Convert scaled arrays back to DataFrames for easier handling
X_train_mental_scaled = pd.DataFrame(X_train_mental_scaled, columns=X_mental.columns, index=X_train_mental.index)
X_test_mental_scaled = pd.DataFrame(X_test_mental_scaled, columns=X_mental.columns, index=X_test_mental.index)

print("Shape of X_train_mental_scaled:", X_train_mental_scaled.shape)
print("Shape of X_test_mental_scaled:", X_test_mental_scaled.shape)
print("Shape of y_train_mental:", y_train_mental.shape)
print("Shape of y_test_mental:", y_test_mental.shape)

print("\nFirst 5 rows of scaled X_train_mental:")
display(X_train_mental_scaled.head())

Shape of X_train_mental_scaled: (2400, 16)
Shape of X_test_mental_scaled: (600, 16)
Shape of y_train_mental: (2400, 1)
Shape of y_test_mental: (600, 1)

First 5 rows of scaled X_train_mental:


,age,sleep_duration,work_hours_per_week,screen_time,sleep_efficiency,gender_Female,gender_Male,gender_Other,exercise_level_High,exercise_level_Low,exercise_level_Moderate,mental_health_condition_Anxiety,mental_health_condition_Bipolar,mental_health_condition_Depression,mental_health_condition_No,mental_health_condition_Ptsd
642,-0.167188,0.615421,1.715896,-1.015081,0.615421,-0.719074,-0.699166,1.422206,-0.687965,-0.725084,1.411566,1.942002,-0.47578,-0.485633,-0.504553,-0.518816
700,0.802557,-1.537556,1.628398,-0.615994,-1.537556,-0.719074,1.430275,-0.703133,-0.687965,1.379150,-0.708433,-0.514933,-0.47578,-0.485633,-0.504553,1.927467
226,-0.987741,-0.932032,0.840922,1.151388,-0.932032,-0.719074,1.430275,-0.703133,1.453561,-0.725084,-0.708433,1.942002,-0.47578,-0.485633,-0.504553,-0.518816
1697,-1.733699,0.009896,-0.121548,0.011141,0.009896,-0.719074,1.430275,-0.703133,-0.687965,-0.725084,1.411566,-0.514933,-0.47578,-0.485633,1.981951,-0.518816
1010,0.280387,1.826470,-0.821527,-0.730019,1.826470,1.390678,-0.699166,-0.703133,-0.687965,-0.725084,1.411566,-0.514933,-0.47578,2.059168,-0.504553,-0.518816


Melakukan train test split 80:20 dan melakukan scalling pada variabel x

### Export Data

In [26]:
# Create the transformed directory if it doesn't exist
os.makedirs(TRANS_PATH, exist_ok=True)

# Save sleep data
X_train_sleep_smote.to_csv(os.path.join(TRANS_PATH, 'X_train_sleep_smote.csv'), index=False)
X_test_sleep_scaled.to_csv(os.path.join(TRANS_PATH, 'X_test_sleep_scaled.csv'), index=False)
y_train_sleep_smote.to_csv(os.path.join(TRANS_PATH, 'y_train_sleep_smote.csv'), index=False)
y_test_sleep.to_csv(os.path.join(TRANS_PATH, 'y_test_sleep.csv'), index=False)

# Save mental health data
X_train_mental_scaled.to_csv(os.path.join(TRANS_PATH, 'X_train_mental_scaled.csv'), index=False)
X_test_mental_scaled.to_csv(os.path.join(TRANS_PATH, 'X_test_mental_scaled.csv'), index=False)
y_train_mental.to_csv(os.path.join(TRANS_PATH, 'y_train_mental.csv'), index=False)
y_test_mental.to_csv(os.path.join(TRANS_PATH, 'y_test_mental.csv'), index=False)

print(f"[SUCCESS] Transformed sleep and mental health data saved to: {TRANS_PATH}")

[SUCCESS] Transformed sleep and mental health data saved to: /content/drive/MyDrive/DeepWell/data/transformed/


In [27]:
# Create the transformed directory if it doesn't exist (already done in previous cell, but good practice)
os.makedirs(TRANS_PATH, exist_ok=True)

# Save sleep data as .pkl
X_train_sleep_smote.to_pickle(os.path.join(TRANS_PATH, 'X_train_sleep_smote.pkl'))
X_test_sleep_scaled.to_pickle(os.path.join(TRANS_PATH, 'X_test_sleep_scaled.pkl'))
y_train_sleep_smote.to_pickle(os.path.join(TRANS_PATH, 'y_train_sleep_smote.pkl'))
y_test_sleep.to_pickle(os.path.join(TRANS_PATH, 'y_test_sleep.pkl'))

# Save mental health data as .pkl
X_train_mental_scaled.to_pickle(os.path.join(TRANS_PATH, 'X_train_mental_scaled.pkl'))
X_test_mental_scaled.to_pickle(os.path.join(TRANS_PATH, 'X_test_mental_scaled.pkl'))
y_train_mental.to_pickle(os.path.join(TRANS_PATH, 'y_train_mental.pkl'))
y_test_mental.to_pickle(os.path.join(TRANS_PATH, 'y_test_mental.pkl'))

print(f"[SUCCESS] Transformed sleep and mental health data saved as .pkl to: {TRANS_PATH}")

[SUCCESS] Transformed sleep and mental health data saved as .pkl to: /content/drive/MyDrive/DeepWell/data/transformed/


In [28]:

pickle.dump(scaler_sleep, open(os.path.join(TRANS_PATH, 'scaler_sleep.pkl'), 'wb'))

In [29]:
scaler_sleep

StandardScaler()

## insight

1. Data yang digunakan unutk train adalah Sleep dan Mental. Data Stress tidak digunakan karena input user tidak sesuai dengan fitur yang tersedia, bisa dipertimbangkan jika ditambahakan input pada frontend
2. Data Sleep menggunakan fitur 	gender, age,	occupation,	sleep_duration	sleep_quality,	stress_level,	bmi_cat,	sleep_disorder dan dibutuhkan feature enggineering pada backend unutk menghitung BMI user agar sesuai dangan data train dan setelah itu dilakukan encoding
3. Data Mental menggunakan fitur 	age,	gender,	exercise_level,	sleep_duration,	mental_health_condition,	work_hours_per_week,	screen_time,	happiness_score dan setelah itu dilakukan encoding
4. Untuk train test data dibagi 80:20 dan untuk taget data sleep itu sleep_disorder yang dimana itu klasifikasi gangguan tidur. Untuk target data mental adalah happiness_score yng dimana merupakan regressi
5. Melakukan scalling pada fitur X untuk masuk model
6. Kemungkinan besar dataset stress tidak digunakan karena fitur input yang tersedia tidak sesuai dengan data train. Jika dataset tersebut tetap ingin digunakan, maka perlu dipertimbangkan penambahan beberapa input pada frontend agar sesuai dengan fitur yang dibutuhkan model.


Data user input

## Dataset 1 — Sleep

| Kolom | Tipe Data | Jenis Data | Penjelasan |
|---|---|---|---|
| `gender` | Object/String | Kategorikal Nominal | Jenis kelamin, misalnya Male atau Female |
| `age` | Integer | Numerik Diskrit | Umur individu dalam tahun |
| `occupation` | Object/String | Kategorikal Nominal | Jenis pekerjaan, misalnya Nurse, Doctor, Engineer |
| `sleep_duration` | Float | Numerik Kontinu | Lama tidur dalam jam |
| `sleep_quality` | Integer | Numerik Ordinal | Tingkat kualitas tidur berdasarkan skala tertentu |
| `stress_level` | Integer | Numerik Ordinal | Tingkat stres berdasarkan skala tertentu |


---

## Dataset 2 — Mental

| Kolom | Tipe Data | Jenis Data | Penjelasan |
|---|---|---|---|
| `age` | Integer | Numerik Diskrit | Umur individu dalam tahun |
| `gender` | Object/String | Kategorikal Nominal | Jenis kelamin |
| `exercise_level` | Object/String | Kategorikal Ordinal | Tingkat olahraga seperti Low, Moderate, dan High |
| `sleep_duration` | Float | Numerik Kontinu | Durasi tidur dalam jam |
| `mental_health_condition` | Object/String | Kategorikal Nominal | Kondisi kesehatan mental seperti Anxiety, Depression, dan PTSD |
| `work_hours_per_week` | Integer | Numerik Diskrit | Jumlah jam kerja per minggu |
| `screen_time` | Float | Numerik Kontinu | Lama penggunaan layar per hari dalam jam |

## For AI

1. Untuk hasil prediksi Happiness Score lakukan normalisasi karena itu regressi
2. Untuk hasil klasifikasi Sleep Disorder kalukan prediksi probabilitas
3. Lakukan Agregasi untuk menghitung wellbeing score

## Referensi Metode Wellbeing Score

Perhitungan *Wellbeing Score* pada penelitian ini menggunakan pendekatan **Composite Indicator** dengan metode **weighted additive aggregation**. Metode ini menggabungkan beberapa indikator menjadi satu skor komposit menggunakan bobot tertentu sesuai tingkat kepentingannya.

Rumus umum *composite indicator* menurut OECD adalah:

$$
CI = \sum_{i=1}^{n} w_i x_i
$$

dengan:

- $CI$ = *Composite Index*  
- $w_i$ = bobot indikator ke-$i$  
- $x_i$ = nilai indikator ke-$i$  

Pada penelitian ini, *Wellbeing Score* dihitung menggunakan kombinasi antara nilai kebahagiaan yang telah dinormalisasi dan skor kesehatan tidur (*Sleep Health Score*):

$$
Wellbeing = 0.6(H_{norm}) + 0.4(SleepHealth)
$$

Keterangan:

- $H_{norm}$ = *Happiness Score* yang telah dinormalisasi  
- $SleepHealth$ = skor kesehatan tidur  
- Bobot 0.6 diberikan pada *Happiness* karena dianggap sebagai indikator utama kesejahteraan  
- Bobot 0.4 diberikan pada *Sleep Health* sebagai faktor pendukung kesejahteraan  

Pendekatan ini mengacu pada konsep *weighted aggregation* yang dijelaskan dalam handbook OECD mengenai pembangunan indikator komposit.

---

## Referensi

Organisation for Economic Co-operation and Development (OECD). (2008). *Handbook on Constructing Composite Indicators: Methodology and User Guide*. OECD Publishing.  

https://doi.org/10.1787/9789264043466-en

## For FS

1. Menambahkan feature engineering pada variabel BMI untuk menghitung manual BMI user
2. Menambahkan input baru: occupation, work_hours_per_week, dan mental_health_condition
3. Disarankan mengganti activity level menjadi Exercise Level karena pada dataset training tidak terdapat fitur activity level
4. Untuk Feature engineering dilakukan dibalik layar